In [ ]:
import pandas as pd
from pathlib import Path

LOADING DATA

In [ ]:
# parquet file storage for more compressed representation vs JSON
data_path = Path.cwd() / "raw_overflow_data.parquet"
print(f"Data exist: {data_path.exists()}")
print(f"Check if is a file: {data_path.is_file()}")

In [ ]:
# extracting the data with pandas
df = pd.read_parquet(data_path)
df.head()

In [ ]:
print(f"Null values check.")
df.isna().sum()

In [ ]:
# all topics
topics = df['topic'].unique()
topics

In [ ]:
# 3000 was the max amount of data pulled per topic, trying to see if there's undersampled classes

def topics_under_size(df, counts=3000):
    
    topic_counts = df.loc[:,'topic'].value_counts()
    topic_counts_filter = topic_counts < counts
    return topic_counts[topic_counts_filter].sort_values()

topics_under_size(df)

In [ ]:
# we can fix some of these with mapping close topics together.
topic_map = {
    'COMPOSITION' : 'OOP_CONCEPTS',
    'AGGREGATION' : 'OOP_CONCEPTS',
    'ENCAPSULATION' : 'OOP_CONCEPTS',
    'MINIMAL AND COMPLETE CLASSES' : 'OOP_CONCEPTS',
    'GRAPH THEORY' : 'ADVANCED_TREES_GRAPHS',
    'MULTIWAY TREES' : 'ADVANCED_TREES_GRAPHS',
}

# while we can't get all classes to 3k we can get close
# during training, we can class weights to give give classes equal influences during training
mapped_df = df.copy()
mapped_df['topic'] = df['topic'].replace(topic_map)

topics_under_size(mapped_df)

VISUALIZING DATA (KINDA)

In [ ]:

def random_sample_topics(df, number_per_topic=2):
    
    # finding the count for smallest group
    smallest_topic = df['topic'].value_counts().min()
        
    # randomly sample 
    random_samples = df.groupby('topic').sample(n=number_per_topic, random_state=42)

    return random_samples.drop(columns=['question_title'])
    
random_sample_topics(mapped_df)

NOTE: Due to the nature of our data (heavily code-based), common cleaning techniques can serve more to hinder than help. According to the average sample, there are a lot of new lines. These are likely the things that need the most attention.

In [ ]:
import html

cleaned_df = mapped_df.copy()

# decode html code back into regular characters
cleaned_df = cleaned_df.apply(html.unescape)

# if the string pattern matches newline or carriage return (\r) ONE or MORE time, replace with space
cleaned_df = cleaned_df.replace(r'[\n\r]+', ' ', regex=True)

In [ ]:
random_sample_topics(cleaned_df)

In [ ]:
topic_label_map = {key : value for value, key in enumerate(cleaned_df['topic'].value_counts().index)}
topic_label_map

In [ ]:
# label mapping for ML work
cleaned_df['label'] = cleaned_df['topic'].map(topic_label_map)

# seeing the labels worked as intended (all true means nothing was mislabled)
cleaned_df['topic'].value_counts().values == cleaned_df['label'].value_counts().values

In [ ]:
# saving to parquet
save_path = Path.cwd() / "train_ready.parquet"
cleaned_df.to_parquet(path=save_path)

if save_path.is_file():
    
    try:
        test = pd.read_parquet(save_path)
        
        print('If all zero, then file integrity is good.')
        print((test != cleaned_df).sum())
        
        print(f'Successfully saved: {save_path}')
        
    except Exception as e:
        print('Save Failed.', {e})
else:
    print('Save failed.')